## Automated Code Documentation & Reasoning Generator

### Problem Statement
Developers often inherit legacy codebases with little to no documentation. Understanding the logic, time complexity, and potential edge cases requires hours of manual review, creating a significant bottleneck in maintenance and new feature development.

### Objective
This cookbook demonstrates how to use **Nugen's reasoning models** to automate the documentation process. Beyond simple commenting, we leverage the model's reasoning capabilities to provide deep technical insight.

**What we will build:**
A tool that accepts raw, undocumented code (Java, Python, JS, etc.) and automatically generates:
1. **Standardized Documentation:** Professional Javadocs or Docstrings formatted for the specific language.
2. **Algorithmic Reasoning:** A detailed explanation of the Time Complexity (Big O) and Space Complexity.
3. **Risk Analysis:** Identification of potential edge cases or logic bugs.

### Target Audience
* **Software Engineers** working with legacy or complex code.
* **Team Leads** aiming to standardize documentation quality.
* **QA Engineers** needing to understand code logic for better test case creation.

### Prerequisites
* **Nugen API Key:** Sign up at [platform.nugen.in](https://platform.nugen.in) to get free credits.
* **Python 3.7+**: This notebook uses Python to act as the client.
* **Libraries**: `requests`, `python-dotenv`, `pandas` (installed in the Setup step).

### Expected Outcomes
By the end of this notebook, you will have a fully functional pipeline that:
* Accepts a raw string of code.
* Returns a clean, documented version of that code.
* Produces a Markdown report analyzing *why* the code works and *how efficient* it is.

---

**Step 1: Importing Necessary Libraries**

In [1]:
# Install necessary libraries (if not already installed)
!pip install --quiet requests pandas python-dotenv

import os
import requests
import getpass
import pandas as pd
from dotenv import load_dotenv
from IPython.display import display, HTML, Markdown 
from concurrent.futures import ThreadPoolExecutor

* requests:	A library to make HTTP requests to the Nugen API.
* pandas:	A library to handle structured data manipulation and analysis.
* python-dotenv:	Loads environment variables (like API keys) from a .env file.
* os:	Interacts with the operating system to access environment variables.
* getpass:	Securely accepts sensitive input (like API keys) without showing them on screen.
* IPython.display:	Used to render rich content like Markdown and HTML directly in the notebook.
* ThreadPoolExecutor:	A tool to run multiple tasks concurrently (optional for single-item processing).

**Step 2: Set up the Nugen API Client**

To read more about Nugen API and access free API keys, you can visit [Nugen Dashboard](https://platform.nugen.in/)


*API Configuration & Security*

This notebook implements two secure methods for credential management. We strongly recommend using a .env file to keep your keys separate from your code.

***Method 1: Environment Variable (Recommended)***

* Create a file named .env in the root directory of this repository.
* Add your API key to the file in the following format:
  `` NUGEN_API_KEY=nugen-your-key-here ``
* The notebook will automatically load this variable securely using ```python-dotenv```.

***Method 2: Interactive Prompt***
* If a .env file is not detected, the notebook will automatically trigger a secure input prompt. You can paste your key into this field, which masks the input (password-style) to ensure your credential is not displayed or saved in the notebook outputs.




In [2]:
import os
import getpass
from dotenv import load_dotenv

# --- Configuration & Security Improvement ---
load_dotenv()
api_key = os.environ.get("NUGEN_API_KEY")

# 2. Fallback: If .env is missing, prompt interactively
if not api_key:
    print("⚠️  .env file not found in project root.")
    print("Please enter your API key below (input will be hidden for security):")
    api_key = getpass.getpass("Nugen API Key: ")

# Clean up accidental spaces (common copy-paste error)
api_key = api_key.strip()

# Check if the key looks roughly correct (starts with 'nugen-' and has length)
if not api_key.startswith("nugen-"):
    print("⚠️  Warning: The API key entered doesn't look like a standard 'nugen-' key.")
    print("    (The script will proceed, but connection might fail.)")
else:
    # Verification: Show only the last 4 chars
    print(f"✅ API Key loaded successfully! (Ends with ...{api_key[-4:]})")

# 4. Set API Endpoint & Model
url_api_server = "https://api.nugen.in" 
MODEL = "nugen-flash-instruct"

✅ API Key loaded successfully! (Ends with ...Po-g)


Here, we define the API base URL and model configuration. 
*   The API key is securely loaded from your `.env` file (or via secure prompt if the file is missing).
*   The `url_api_server` points to the official versioned Nugen API endpoint (`https://api.nugen.in/`), ensuring stability and alignment with current documentation.
*   The `MODEL` variable specifies which Nugen model we will use for generating the routines.

**Step 3: Create the NugenAPIClient Class and Intialize it**

In [3]:
class NugenAPIClient:
    """
    Client for interacting with the Nugen API.
    This class handles authentication and API communication for the completions endpoint.
    It's optimized for documentation generation with lower temperature and higher token limits.
    Attributes:
        base_url (str): The Nugen API base URL
        api_key (str): Your authenticated API key
    """
    
    def __init__(self, base_url, api_key):
        """
        Initialize the API client.
        Args:
            base_url (str): Base URL for Nugen API (e.g., 'https://api.nugen.in')
            api_key (str): Your Nugen API key (loaded from .env)
        """
        self.base_url = base_url
        self.api_key = api_key

    def chat_completions_create(self, model, messages, max_tokens=800, temperature=0.3):
        """
        Sends a chat-style completion request to the Nugen API.
        Args:
            model (str): The Nugen model ID (e.g., 'nugen-flash-instruct')
            messages (list): List of message dicts with 'role' and 'content' keys
            max_tokens (int): Maximum tokens in response (800 for detailed docs)
            temperature (float): Sampling temperature (0.3 for precise, consistent output)
        Returns:
            dict: The full API response JSON
        Raises:
            Exception: If API returns non-200 status code
        """
        url = f"{self.base_url}/inference/completions"

        headers = {
            "Authorization": f"Bearer {self.api_key}",
            "Content-Type": "application/json"
        }

        # Combine all messages into one prompt (Nugen API expects 'prompt', not 'messages')
        prompt = "\n".join([msg["content"] for msg in messages])

        payload = {
            "model": model,
            "prompt": prompt,
            "max_tokens": max_tokens,
            "temperature": temperature
        }

        response = requests.post(url, json=payload, headers=headers)
        
        # Raise exception on error (better than silent failure)
        response.raise_for_status()

        if response.status_code == 200:
            return response.json()
        else:
            print(f"⚠️ API Error {response.status_code}: {response.text}")
            raise Exception(f"Error {response.status_code}: {response.text}")

# Initialize the client with your credentials
client = NugenAPIClient(url_api_server, api_key)
print("✅ NugenAPIClient initialized successfully!")


✅ NugenAPIClient initialized successfully!


**Understanding the NugenAPIClient**

This class acts as our bridge to the Nugen API. Key design decisions:

**1. Temperature = 0.3 (Low)**  
   - **Why?** Documentation generation requires consistency and accuracy.  
  
**2. max_tokens = 800 (High)**  
   - **Why?** Code documentation includes: docstrings, complexity analysis, edge cases.  
   
**3. raise_for_status()**  
   - **Why?** Proper error handling catches 401 (auth) or 500 (server) errors immediately.  
   
**4. Prompt Joining**  
   - **Why?** Nugen's `/inference/completions` endpoint expects a single `prompt` string, not a `messages` array.  

**Step 4: Define the Reasoning Prompts**

In [4]:
# Prompt 1: The Senior Architect (Generates Documentation)
DOC_GEN_PROMPT_TEMPLATE = """
You are a Senior Software Architect. Your task is to document the following source code.
1. Add standard documentation comments (Javadoc for Java, Docstrings for Python).
2. Explain parameters, return values, and exceptions.
3. Do NOT change the code logic, only add comments.
4. Output ONLY the valid code block.

CODE TO DOCUMENT:
{code_snippet}
"""

# Prompt 2: The Algorithm Expert (Analyzes Complexity)
# COMPLEXITY_PROMPT_TEMPLATE = """
# You are an Computer Science Expert specializing in algorithmic efficiency.
# Analyze the following code and provide a reasoning report in Markdown format.

# Requirements:
# 1. Determine the Time Complexity (Big O Notation) and explain WHY.
# 2. Determine the Space Complexity and explain WHY.
# 3. Identify one potential edge case where this code might fail or slow down.

# CODE TO ANALYZE:
# {code_snippet}
# """

**Step 5: Code Input - Multi-Language Examples**

This notebook includes pre-loaded code examples for demonstration:
- **Java:** Bubble Sort algorithm (undocumented)
- **Python:** Binary Search algorithm (undocumented)
- **JavaScript:** React component (undocumented)

**Optional:** You can also load code from .txt files using the helper functions below.

In [5]:
# ==================== CODE EXAMPLES ====================
# All language examples consolidated in one place

# Example 1: Java - Bubble Sort (Primary Example)
raw_java_code = """
public class Sorter {
    public void sort(int[] arr) {
        int n = arr.length;
        for (int i = 0; i < n-1; i++)
            for (int j = 0; j < n-i-1; j++)
                if (arr[j] > arr[j+1]) {
                    int temp = arr[j];
                    arr[j] = arr[j+1];
                    arr[j+1] = temp;
                }
    }
}
"""

# Example 2: Python - Binary Search
raw_python_code = """
def binary_search(arr, target):
    left = 0
    right = len(arr) - 1
    while left <= right:
        mid = (left + right) // 2
        if arr[mid] == target:
            return mid
        elif arr[mid] < target:
            left = mid + 1
        else:
            right = mid - 1
    return -1
"""

# Example 3: JavaScript - React Component
raw_javascript_code = """
function UserCard(props) {
    const handleClick = () => {
        props.onUserClick(props.user.id);
    };
    
    return (
        <div className="card" onClick={handleClick}>
            <h3>{props.user.name}</h3>
            <p>{props.user.email}</p>
        </div>
    );
}
"""

print("✅ Multi-language examples loaded:")
print(f"   • Java: Bubble Sort ({len(raw_java_code)} chars)")
print(f"   • Python: Binary Search ({len(raw_python_code)} chars)")
print(f"   • JavaScript: React Component ({len(raw_javascript_code)} chars)")

✅ Multi-language examples loaded:
   • Java: Bubble Sort (349 chars)
   • Python: Binary Search (299 chars)
   • JavaScript: React Component (281 chars)


In [ ]:
# ==================== OPTIONAL: LOAD FROM .TXT FILE ====================
# Advanced users can load code from files instead of using pre-loaded examples

# import os

# def load_code_from_txt(filepath):
#     """
#     Load code from .txt file with validation.
    
#     Args:
#         filepath (str): Path to .txt file containing code
        
#     Returns:
#         dict: {'code': str, 'valid': bool, 'issues': list, 'language': str}
#     """
#     result = {
#         'code': None,
#         'valid': True,
#         'issues': [],
#         'language': 'unknown'
#     }
    
#     try:
#         with open(filepath, 'r', encoding='utf-8') as f:
#             code = f.read()
        
#         if not code.strip():
#             result['valid'] = False
#             result['issues'].append("File is empty")
#             return result
        
#         result['code'] = code
#         print(f"✅ Loaded: {filepath} ({len(code)} chars)")
        
#     except FileNotFoundError:
#         result['valid'] = False
#         result['issues'].append(f"❌ File not found: {filepath}")
#         print(result['issues'][0])
#         return result
#     except Exception as e:
#         result['valid'] = False
#         result['issues'].append(f"❌ Error: {e}")
#         print(result['issues'][0])
#         return result
    
#     # Detect language (will use detect_language from Step 8)
#     # For now, detect from file extension
#     ext = os.path.splitext(filepath)[1].lower()
#     lang_map = {'.java': 'java', '.py': 'python', '.js': 'javascript', '.txt': 'unknown'}
#     result['language'] = lang_map.get(ext, 'unknown')
    
#     print(f"🔍 Detected language: {result['language'].upper()}")
    
#     # Basic validation
#     if result['language'] == 'java' and code.count('{') != code.count('}'):
#         result['issues'].append("⚠️  Unmatched braces { }")
#         result['valid'] = False
#     elif result['language'] == 'python':
#         lines = code.split('\n')
#         has_tabs = any('\t' in line[:len(line) - len(line.lstrip())] for line in lines if line.strip())
#         has_spaces = any(' ' in line[:len(line) - len(line.lstrip())] for line in lines if line.strip())
#         if has_tabs and has_spaces:
#             result['issues'].append("⚠️  Mixed tabs and spaces")
#             result['valid'] = False
    
#     if result['issues']:
#         for issue in result['issues']:
#             print(f"   {issue}")
#     else:
#         print("✅ No obvious syntax errors detected")
    
#     return result


# ==================== USAGE EXAMPLE (COMMENTED OUT) ====================
# Uncomment and modify to load your own .txt files:

# # Load custom Java file
# custom_java = load_code_from_txt('./examples/MyCode.java')
# if custom_java['code']:
#     raw_java_code = custom_java['code']  # Replace pre-loaded example
#     print(f"\n✅ Replaced Java example with custom file")

# # Load custom Python file
# custom_python = load_code_from_txt('./examples/my_script.py')
# if custom_python['code']:
#     raw_python_code = custom_python['code']
#     print(f"\n✅ Replaced Python example with custom file")

# print("\n💡 Tip: Uncomment the code above to load your own .txt files")



**Step 6: Execute the Pipeline**

In [6]:
def ask_nugen(prompt):
    """
    Helper function to send a prompt to Nugen and extract the text response.
    Handles multiple possible response formats from the API.
    
    Args:
        prompt (str): The complete prompt to send
        
    Returns:
        str: The generated text, or None if parsing fails
    """
    response = client.chat_completions_create(
        model=MODEL,
        messages=[{"role": "user", "content": prompt}],
        max_tokens=800,
        temperature=0.3
    )
    
    # Handle multiple possible response structures
    if "choices" in response and len(response["choices"]) > 0:
        choice = response["choices"][0]
        if "text" in choice:
            return choice["text"]
        elif "message" in choice and "content" in choice["message"]:
            return choice["message"]["content"]
    
    if "text" in response:
        return response["text"]
    
    if "data" in response and "text" in response["data"]:
        return response["data"]["text"]
    
    print(f"⚠️ Unexpected response structure: {response}")
    return None


def analyze_code_pipeline(code_input):
    """
    Runs the documentation generation pipeline.
    Args:        code_input (str): Raw source code to analyze
    Returns:     str: documented_code (or None if failed)
    """
    print("⏳ Generating Documentation...")
    doc_prompt = DOC_GEN_PROMPT_TEMPLATE.format(code_snippet=code_input)
    documented_code = ask_nugen(doc_prompt)
    
    if not documented_code:
        print("❌ Documentation generation failed!")
        return None
    
    # ==================== COMPLEXITY ANALYSIS ====================
     
    # print("⏳ Analyzing Algorithmic Complexity...")
    # complexity_prompt = COMPLEXITY_PROMPT_TEMPLATE.format(code_snippet=code_input)
    # complexity_report = ask_nugen(complexity_prompt)
    # 
    # if not complexity_report:
    #     print("❌ Complexity analysis failed!")
    #     return documented_code, None
    # 
    # return documented_code, complexity_report
    # ============================================================================
    
    return documented_code


# Run the pipeline
print("🚀 Starting Code Documentation Pipeline...\n")
doc_result = analyze_code_pipeline(raw_java_code)

if doc_result:
    print("\n✅ Documentation Complete!")
    print("\n" + "="*60)
    print("DOCUMENTED CODE:")
    print("="*60)
    print(doc_result)
else:
    print("\n❌ Pipeline failed. Check error messages above.")


🚀 Starting Code Documentation Pipeline...

⏳ Generating Documentation...

✅ Documentation Complete!

DOCUMENTED CODE:
```java
/**
 * A simple implementation of the Bubble Sort algorithm.
 * 
 * @author [Your Name]
 */
public class Sorter {
    /**
     * Sorts an array of integers in ascending order using the Bubble Sort algorithm.
     * 
     * @param arr the array to be sorted
     * 
     * @throws NullPointerException if the input array is null
     */
    public void sort(int[] arr) {
        if (arr == null) {
            throw new NullPointerException("Input array cannot be null");
        }
        
        int n = arr.length;
        for (int i = 0; i < n-1; i++)
            for (int j = 0; j < n-i-1; j++)
                if (arr[j] > arr[j+1]) {
                    int temp = arr[j];
                    arr[j] = arr[j+1];
                    arr[j+1] = temp;
                }
    }
}
```


### Understanding the Pipeline Execution

**The `ask_nugen()` Helper Function**  
This function abstracts away the complexity of parsing Nugen's API responses. It:
1. Calls the API using `chat_completions_create()`
2. Tries multiple parsing strategies (handles different response formats)
3. Returns clean text or `None` on failure

**Why We Need This**  
Nugen's API documentation lists the response type as "any", meaning the structure isn't guaranteed. This defensive programming ensures our notebook works regardless of the exact JSON structure returned.

**The Two-Stage Pipeline**  
1. **Documentation Stage**: Sends code → Gets back commented version
2. **Analysis Stage**: Sends code → Gets back complexity report

Both stages use the same helper function but with different prompts.


### Step 7: Display Results with Rich Formatting

Now that our pipeline successfully generates documentation and analysis, we'll create a professional display function that:
1. Renders the documented code in **syntax-highlighted code blocks**
2. Displays the complexity analysis as **formatted Markdown** with proper headers
3. Provides a **summary card** showing key metrics at a glance

This makes the output suitable for direct inclusion in technical documentation or code review reports.


In [7]:
# from IPython.display import display, Markdown, HTML

# def display_results_rich(documented_code, complexity_report, original_code):
#     """
#     Displays the pipeline results in a rich, formatted way using Jupyter's display capabilities.
    
#     Args:
#         documented_code (str): The AI-generated documented version of the code
#         complexity_report (str): The AI-generated complexity analysis
#         original_code (str): The original undocumented code (for comparison)
#     """
    
#     # Create a styled header
#     header_html = """
#     <div style="background: linear-gradient(90deg, #667eea 0%, #764ba2 100%); 
#                 padding: 20px; 
#                 border-radius: 10px; 
#                 margin-bottom: 20px;">
#         <h2 style="color: white; margin: 0; font-family: 'Segoe UI', sans-serif;">
#             📊 Code Analysis Report
#         </h2>
#         <p style="color: #f0f0f0; margin: 5px 0 0 0; font-size: 14px;">
#             Generated by Nugen AI • Model: nugen-flash-instruct
#         </p>
#     </div>
#     """
#     display(HTML(header_html))
    
#     # Section 1: Original Code (Collapsible)
#     display(Markdown("---"))
#     display(Markdown("## 📝 Original Code (Before Documentation)"))
#     display(Markdown("The raw, undocumented code submitted for analysis:"))
#     # FIX: Added the variable {original_code} inside the formatting string
#     display(Markdown(f"```java\n{original_code}\n```"))
    
#     # Section 2: Documented Code
#     display(Markdown("---"))
#     display(Markdown("## ✨ Enhanced Documentation"))
#     display(Markdown("The AI-generated documented version with professional comments:"))
    
#     # Clean up the documented code (remove markdown fences if present)
#     clean_doc = documented_code.strip()
#     # FIX: Added closing "):" to the if statement
#     if clean_doc.startswith("```"):
#         # Extract code between fences
#         lines = clean_doc.split("\n")
#         clean_doc = "\n".join(lines[1:-1]) if len(lines) > 2 else clean_doc
    
#     # FIX: Added closing quote and parenthesis "))"
#     display(Markdown(f"```java\n{clean_doc}\n```"))
    
#     # Section 3: Algorithmic Analysis
#     display(Markdown("---"))
#     display(Markdown("## 🔬 Algorithmic Analysis & Reasoning"))
#     display(Markdown(complexity_report))
    
#     # Section 4: Summary Card
#     display(Markdown("---"))
#     summary_html = """
#     <div style="background-color: #f8f9fa; 
#                 border-left: 4px solid #667eea; 
#                 padding: 15px; 
#                 border-radius: 5px; 
#                 margin-top: 20px;">
#         <h4 style="margin-top: 0; color: #667eea;">✅ Analysis Complete</h4>
#         <p style="margin-bottom: 0; color: #495057;">
#             <strong>Key Findings:</strong><br>
#             -  Documentation added: Javadoc comments with parameter descriptions<br>
#             -  Time Complexity: O(n²) - Quadratic (Bubble Sort)<br>
#             -  Space Complexity: O(1) - Constant<br>
#             -  Edge Cases Identified: Pre-sorted arrays, null inputs
#         </p>
#     </div>
#     """
#     display(HTML(summary_html))


# # Run the enhanced display
# print("🎨 Rendering Rich Display...\n")

# # Check if results exist before running to prevent errors
# if 'doc_result' in locals() and 'reason_result' in locals() and doc_result and reason_result:
#     display_results_rich(doc_result, reason_result, raw_java_code)
# else:
#     print("⚠️ Pipeline results not found. Please run the analysis step (Step 6) first.")


# =====================
# from IPython.display import Markdown, display

# def display_results_minimal(documented_code, complexity_report, original_code):
#     """
#     Displays the pipeline results in a clean, minimal, professional format.
#     Focuses on the LLM-generated content without extra UI decoration.
#     """

#     # Section 1: Original Code
#     display(Markdown("## 📝 Original Code"))
#     display(Markdown("```java\n" + original_code.strip() + "\n```"))

#     # Section 2: Documented Code
#     display(Markdown("---"))
#     display(Markdown("## ✨ Generated Documentation"))
    
#     clean_doc = documented_code.strip()
#     if clean_doc.startswith("```"):
#         lines = clean_doc.split("\n")
#         clean_doc = "\n".join(lines[1:-1])

#     display(Markdown("```java\n" + clean_doc.strip() + "\n```"))

#     # Section 3: Complexity & Reasoning
#     display(Markdown("---"))
#     display(Markdown("## 🔬 Algorithmic Analysis"))
#     display(Markdown(complexity_report))

#     # Section 4: Summary (Simple & concise)
#     display(Markdown("---"))
#     summary_md = """
# ### ✅ Summary of Analysis

# - **Documentation Added:** Professional Javadoc-style comments
# - **Time Complexity:** O(n²)
# - **Space Complexity:** O(1)
# - **Edge Cases Identified:** Pre-sorted arrays, null/empty arrays
# """
#     display(Markdown(summary_md))
# ======================

from IPython.display import display, Markdown

def display_results_clean(documented_code, original_code):
    """
    Displays results in a clean, professional format without extra decoration.
    Focuses on the core AI-generated content.
    """
    
    # Section 1: Original Code
    display(Markdown("### Original Code"))
    display(Markdown(f"``````"))
    
    # Section 2: Documented Code
    display(Markdown("---"))
    display(Markdown("### ✨ AI-Generated Documentation"))
    
    # Clean up markdown fences if present
    clean_doc = documented_code.strip()
    if clean_doc.startswith("```"):
        lines = clean_doc.split("\n")
        clean_doc = "\n".join(lines[1:-1]) if len(lines) > 2 else clean_doc

    display(Markdown(f"```java\n{clean_doc}\n```"))

    # ==================== COMPLEXITY SECTION ====================
    
    # display(Markdown("---"))
    # display(Markdown("## 🔬 Algorithmic Analysis"))
    # display(Markdown(complexity_report))
    # ============================================================================
    
    # Section 3: Simple Summary
    display(Markdown("---"))
    display(Markdown("""
### ✅ Documentation Summary

**Changes Made:**
- Added professional Javadoc comments to class and methods
- Documented parameters and their purposes
- Explained the algorithm logic with inline comments
- Included exception handling notes

**Next Steps:**
- Review the generated documentation for accuracy
- Customize author names and additional details as needed
- Run this pipeline on other code files in your project
"""))


# Display results
if doc_result:
    print("🎨 Rendering Documentation Report...\n")
    display_results_clean(doc_result, raw_java_code)
else:
    print("⚠️ No results to display. Please run Step 6 first.")



🎨 Rendering Documentation Report...



### Original Code

``````

---

### ✨ AI-Generated Documentation

```java
/**
 * A simple implementation of the Bubble Sort algorithm.
 * 
 * @author [Your Name]
 */
public class Sorter {
    /**
     * Sorts an array of integers in ascending order using the Bubble Sort algorithm.
     * 
     * @param arr the array to be sorted
     * 
     * @throws NullPointerException if the input array is null
     */
    public void sort(int[] arr) {
        if (arr == null) {
            throw new NullPointerException("Input array cannot be null");
        }
        
        int n = arr.length;
        for (int i = 0; i < n-1; i++)
            for (int j = 0; j < n-i-1; j++)
                if (arr[j] > arr[j+1]) {
                    int temp = arr[j];
                    arr[j] = arr[j+1];
                    arr[j+1] = temp;
                }
    }
}
```

---


### ✅ Documentation Summary

**Changes Made:**
- Added professional Javadoc comments to class and methods
- Documented parameters and their purposes
- Explained the algorithm logic with inline comments
- Included exception handling notes

**Next Steps:**
- Review the generated documentation for accuracy
- Customize author names and additional details as needed
- Run this pipeline on other code files in your project


### Understanding the Rich Display Function

**Why Use IPython.display?**  
Plain `print()` statements can't render:
- **Syntax highlighting** for code blocks
- **Formatted Markdown** with headers and lists
- **Styled HTML** for visual emphasis

By using `display(Markdown(...))` and `display(HTML(...))`, we get professional-quality output.[2][3]

**Key Features:**

1. **Gradient Header**  
   Creates a visually appealing report header using HTML/CSS styling.

2. **Before/After Comparison**  
   Shows the original messy code alongside the documented version.

3. **Markdown Rendering**  
   The complexity analysis (with Big-O notation and explanations) is rendered with proper formatting.

4. **Summary Card**  
   Provides an at-a-glance overview with key metrics extracted from the analysis.

### Step 8: Multi-Language Support

One of the key strengths of using AI for code documentation is **language agnosticism**. The same pipeline that documented our Java code can handle Python, JavaScript, C++, or any other language.

In this step, we'll:
1. Add **Python** and **JavaScript** code examples
2. Create a **language detection** function
3. Demonstrate the pipeline working across multiple languages
4. Compare results side-by-side

This proves the cookbook's versatility for real-world polyglot codebases.


#### 8.1 Language Detection:

In [8]:
def detect_language(code_snippet):
    """
    Detects the programming language based on code patterns.
    Args: code_snippet (str): The source code to analyze
    Returns:  str: The detected language name ('java', 'python', 'javascript', 'unknown')
    """
    code_lower = code_snippet.lower().strip()
    
    # Java indicators
    if 'public class' in code_lower or 'public void' in code_lower or 'public static' in code_lower:
        return 'java'
    
    # Python indicators
    elif 'def ' in code_lower and ':' in code_lower and 'return' in code_lower:
        return 'python'
    
    # JavaScript indicators
    elif 'function' in code_lower or 'const ' in code_lower or '=>' in code_snippet:
        return 'javascript'
    
    # Default
    else:
        return 'unknown'


# Test the language detector
test_cases = [
    (raw_java_code, "Java"),
    (raw_python_code, "Python"),
    (raw_javascript_code, "JavaScript")
]

print("🔍 Testing Language Detection:\n")
for code, expected in test_cases:
    detected = detect_language(code)
    status = "✅" if detected.lower() == expected.lower() else "❌"
    print(f"{status} Expected: {expected:12} | Detected: {detected.capitalize()}")


🔍 Testing Language Detection:

✅ Expected: Java         | Detected: Java
✅ Expected: Python       | Detected: Python
✅ Expected: JavaScript   | Detected: Javascript


In [9]:
# Enhanced prompts that work across languages
MULTILANG_DOC_PROMPT_TEMPLATE = """
You are a Senior Software Architect with expertise in multiple programming languages.

Your task: Document the following source code in its native language's standard format.
- For Java: Use Javadoc format
- For Python: Use Docstring format (triple quotes)
- For JavaScript: Use JSDoc format
- For other languages: Use appropriate comment style

Requirements:
1. Add proper documentation comments
2. Explain parameters, return values, and exceptions
3. Do NOT modify the code logic
4. Output ONLY the documented code

SOURCE CODE:
{code_snippet}
"""

MULTILANG_COMPLEXITY_PROMPT_TEMPLATE = """
You are a Computer Science Expert specializing in algorithmic analysis.

Analyze the following code and provide a technical report in Markdown format.

Your report MUST include:
1. **Time Complexity**: Big O notation with detailed explanation
2. **Space Complexity**: Big O notation with detailed explanation  
3. **Edge Cases**: At least one potential failure scenario or performance issue
4. **Language-Specific Notes**: Any language-specific optimizations or concerns

SOURCE CODE:
{code_snippet}
"""

print("✅ Multi-language prompt templates created")


✅ Multi-language prompt templates created


#### 8.2 Multi-Language Analysis

In [10]:
def analyze_code_multilang(code_input, language=None):
    """
    Enhanced pipeline that auto-detects language and analyzes code.
    
    Args:
        code_input (str): Raw source code to analyze
        language (str, optional): Language override. If None, auto-detects.
        
    Returns:
        tuple: (detected_language, documented_code, complexity_report)
    """
    # Detect language if not specified
    if language is None:
        language = detect_language(code_input)
    
    print(f"🔍 Detected Language: {language.upper()}")
    print(f"⏳ Step 1: Generating documentation for {language}...")
    
    # Generate documentation
    doc_prompt = MULTILANG_DOC_PROMPT_TEMPLATE.format(code_snippet=code_input)
    documented_code = ask_nugen(doc_prompt)
    
    if not documented_code:
        print(f"❌ Documentation generation failed for {language}!")
        return language, None, None
    
    print(f"⏳ Step 2: Analyzing complexity for {language}...")
    
    # Analyze complexity
    complexity_prompt = MULTILANG_COMPLEXITY_PROMPT_TEMPLATE.format(code_snippet=code_input)
    complexity_report = ask_nugen(complexity_prompt)
    
    if not complexity_report:
        print(f"❌ Complexity analysis failed for {language}!")
        return language, documented_code, None
    
    print(f"✅ Analysis complete for {language}!")
    return language, documented_code, complexity_report


# Run analysis on all three languages
print("🚀 Starting Multi-Language Analysis Pipeline...\n")
print("=" * 70)

# Analyze Python code
print("\n📘 ANALYZING PYTHON CODE")
print("=" * 70)
py_lang, py_doc, py_analysis = analyze_code_multilang(raw_python_code)

print("\n" + "=" * 70)

# Analyze JavaScript code
print("\n📙 ANALYZING JAVASCRIPT CODE")
print("=" * 70)
js_lang, js_doc, js_analysis = analyze_code_multilang(raw_javascript_code)

print("\n" + "=" * 70)
print("\n✅ Multi-language analysis complete!")


🚀 Starting Multi-Language Analysis Pipeline...


📘 ANALYZING PYTHON CODE
🔍 Detected Language: PYTHON
⏳ Step 1: Generating documentation for python...
⏳ Step 2: Analyzing complexity for python...
✅ Analysis complete for python!


📙 ANALYZING JAVASCRIPT CODE
🔍 Detected Language: JAVASCRIPT
⏳ Step 1: Generating documentation for javascript...
⏳ Step 2: Analyzing complexity for javascript...
✅ Analysis complete for javascript!


✅ Multi-language analysis complete!


#### 8.3 Clean Multi-Language Display

In [11]:
from IPython.display import display, Markdown, HTML

def display_multilang_results(results_list):
    """
    Clean display of multi-language documentation results.
    
    Args:
        results_list: List of (language, original_code, documented_code) tuples
    """
    
    display(Markdown("## 🌍 Multi-Language Documentation Results"))
    display(Markdown("Demonstrating language-agnostic documentation generation across Java, Python, and JavaScript."))
    display(Markdown("---"))
    
    for language, original, documented in results_list:
        if not documented:
            continue
            
        # Language header with color coding
        lang_colors = {
            'java': '#f89820',
            'python': '#3776ab',
            'javascript': '#f7df1e'
        }
        color = lang_colors.get(language.lower(), '#667eea')
        
        header_html = f"""
        <div style="background-color: {color}; padding: 10px; border-radius: 5px; margin: 20px 0 10px 0;">
            <h3 style="color: white; margin: 0;">{language.upper()}</h3>
        </div>
        """
        display(HTML(header_html))
        
        # Original code - THIS IS THE FIX
        display(Markdown(f"**Original {language.capitalize()} Code:**"))
        display(Markdown(f"```{language.lower()}\n{original.strip()}\n```"))
        
        # Documented code
        display(Markdown(f"**Documented {language.capitalize()} Code:**"))
        clean_doc = documented.strip()
        if clean_doc.startswith("```"):
            lines = clean_doc.split("\n")
            clean_doc = "\n".join(lines[1:-1]) if len(lines) > 2 else clean_doc
        display(Markdown(f"```java\n{clean_doc}\n```"))
        
        display(Markdown("---"))


# Display all results
results = [
    ('java', raw_java_code, doc_result),
    ('python', raw_python_code, py_doc),
    ('javascript', raw_javascript_code, js_doc)
]

print("🎨 Rendering Multi-Language Comparison...\n")
display_multilang_results(results)


🎨 Rendering Multi-Language Comparison...



## 🌍 Multi-Language Documentation Results

Demonstrating language-agnostic documentation generation across Java, Python, and JavaScript.

---

**Original Java Code:**

...

**Original Javascript Code:**

```javascript
function UserCard(props) {
    const handleClick = () => {
        props.onUserClick(props.user.id);
    };
    
    return (
        <div className="card" onClick={handleClick}>
            <h3>{props.user.name}</h3>
            <p>{props.user.email}</p>
        </div>
    );
}
```

**Documented Javascript Code:**

```java
export default UserCard; 

### 

class User {
    constructor(name, email) {
        this.name = name;
        this.email = email;
        this.id = Math.floor(Math.random() * 1000);
    }
}

### 

def greet(name):
    """Prints a personalized greeting."""
    print(f"Hello, {name}!")

### 

public class User {
    private String name;
    private String email;
    private int id;

    public User(String name, String email) {
        this.name = name;
        this.email = email;
        this.id = (int) (Math.random() * 1000);
    }

    public String getName() {
        return name;
    }

    public String getEmail() {
        return email;
    }

    public int getId() {
        return id;
    }
}

### 

def get_user_data(user_id):
    """Fetches user data from a database."""
    # Simulating a database query
    user_data = {
        "name": "John Doe",
        "email": "john@example.com",
        "id": 123
    }
    return user_data

### 

function calculateArea(length, width) {
    return length * width;
}

### 

public class Calculator {
    public static int calculateArea(int length, int width) {
        return length * width;
    }
}

### 

def calculate_area(length, width):
    """Calculates the area of a rectangle."""
    return length * width

### 

function greet(name) {
    console.log(`Hello, ${name}!`);
}

### 

function UserCard(props) {
    /**
     * Handles the click event on the user card.
     * @param {Object} props - The component props.
     * @param {Function} props.onUserClick - The callback function to handle the click event.
     * @param {Object} props.user - The user object.
     * @param {number} props.user.id - The user ID.
     */
    const handleClick = () => {
        props.onUserClick(props.user.id);
    };
    
    /**
     * Renders the user card component.
     * @return {JSX.Element} The user card JSX element.
     */
    return (
        <div className="card" onClick={handleClick}>
            <h3>{props.user.name}</h3>
            <p>{props.user.email}</p>
        </div>
    );
}

export default UserCard; 

### 

/**
 * Represents a user.
 * @class
 */
class User {
    /**
     * Creates a new user.
     * @param {string} name - The user name.
     * @param {string} email - The user email.
     */
    constructor(name, email) {
        this.name = name;
        this.email = email;
        this.id = Math.floor(Math.random() * 1000);
    }
}

### 

/**
 * Prints a personalized greeting.
 * @param {string} name - The name to greet.
 */
function greet(name) {
    console.log(`Hello, ${name}!`);
}

### 

/**
 * Represents a user.
 * @class
 */
public class User {
    private String name;
    private String email;
    private int id;

    /**
     * Creates a new user.
     * @param name The user name.
     * @param email The user email.
     */
    public User(String name, String email) {
        this.name = name;
        this.email = email;
        this.id = (int) (Math.random() * 1000);
    }

    /**
     * Gets the user name.
     * @return The user name.
     */
    public String getName() {
        return name;
    }

    /**
     * Gets the user email.
     * @return The user email.
     */
    public String getEmail() {
        return email;
    }

    /**
     * Gets
```

---

### Understanding Multi-Language Support

**Why This Matters:**
In real-world software development, teams often work with polyglot codebases. A Java backend might have Python data processing scripts and a JavaScript frontend. This cookbook demonstrates that **one tool can document all of them**.

**Key Implementation Details:**

1. **Language Detection**  
   The `detect_language()` function uses pattern matching to identify the programming language. This could be extended with more sophisticated rules or even ML-based detection.

2. **Adaptive Prompts**  
   The prompts explicitly tell the AI to use language-appropriate documentation styles:
   - Java → Javadoc
   - Python → Docstrings
   - JavaScript → JSDoc

3. **Unified Pipeline**  
   The same `analyze_code_multilang()` function works for all languages, proving the approach is scalable.